# Этап 1 аугментации — генерация писем через LLM (vLLM)

**Модель:** `Qwen/Qwen2.5-14B-Instruct-AWQ` (через vLLM)  
**Вход:** `train.csv`  
**Выход:** `train_after_stage1.csv`  

**Цель:** довести малочисленные классы до 20 документов.

**Схема:** для каждого класса < 20 → описание класса → батчевая генерация с запасом (x2) → фильтры → отбор. Повтор при нехватке (до 5 раз).

In [1]:
import os
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["VLLM_USE_V1"] = "0"

## 1. Зависимости

In [2]:
%%capture
!pip install vllm langdetect scikit-learn sentence-transformers

## 2. Параметры и данные

In [3]:
import re, random, gc
import numpy as np, pandas as pd
from pathlib import Path

TRAIN_PATH = Path("/content/train.csv")
OUT_PATH   = Path("/content/train_after_stage1.csv")

TARGET_COUNT = 20
OVERSAMPLE = 2
MAX_RETRIES  = 5
MAX_EXAMPLES_IN_PROMPT = 5
MIN_LENGTH   = 500
SIM_HIGH     = 0.95
SIM_LOW      = 0.50
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# чтение с обработкой кодировки
try:
    df = pd.read_csv(TRAIN_PATH, encoding="utf-8")
except UnicodeDecodeError:
    df = pd.read_csv(TRAIN_PATH, encoding="cp1251")

print(f"Загружено: {len(df)} документов, {df['label'].nunique()} классов")
print(df["label"].value_counts().tail(10))

Загружено: 1404 документов, 36 классов
label
Блок бизнес-директора                                                           6
Проект «Обустройство Интересного лицензионного участка»                         5
Управление коммуникаций                                                         5
Проект "Обустройство площадных объектов НГКМ Поменбше"                          4
Блок исполнительного директора по реализации проекта "Большое месторождение"    3
Проект «Обустройство объектов Новейшей нейти»                                   2
Подразделение по информационным технологиям                                     1
Имущественные вопросы                                                           1
Блок заместителя генерального директора по строительству                        1
Проект «Трубопроводный транспорт Ещё одного НГКМ»                               1
Name: count, dtype: int64


## 3. Загрузка LLM через vLLM

In [4]:
from vllm import LLM, SamplingParams

MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct-AWQ"

llm = LLM(
    model=MODEL_NAME,
    quantization="awq",
    max_model_len=8192,
    gpu_memory_utilization=0.90,
    enforce_eager=True,
    trust_remote_code=True,
)
print("Модель загружена через vLLM")

# параметры генерации писем
gen_params = SamplingParams(
    temperature=0.7, top_p=0.9, top_k=50,
    max_tokens=1024, repetition_penalty=1.1,
)
# параметры для описания класса
ctx_params = SamplingParams(
    temperature=0.3, top_p=0.9, max_tokens=200,
)

INFO 05-21 14:07:44 [utils.py:240] non-default args: {'trust_remote_code': True, 'max_model_len': 8192, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'quantization': 'awq', 'enforce_eager': True, 'model': 'Qwen/Qwen2.5-14B-Instruct-AWQ'}
WARNING 05-21 14:07:44 [envs.py:1866] Unknown vLLM environment variable detected: VLLM_USE_V1


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

INFO 05-21 14:08:04 [model.py:568] Resolved architecture: Qwen2ForCausalLM
INFO 05-21 14:08:04 [model.py:1697] Using max model len 8192
INFO 05-21 14:08:04 [awq_marlin.py:256] Detected that the model can run with awq_marlin, however you specified quantization=awq explicitly, so forcing awq. Use quantization=awq_marlin for faster inference
INFO 05-21 14:08:04 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Модель загружена через vLLM


## 4. Промпты и батчевая генерация

In [5]:
SYSTEM_PROMPT = (
    "Ты пишешь одно входящее электронное письмо на русском языке. "
    "Письмо должно быть развёрнутым и подробным — не менее 200 токенов "
    "(примерно 800 символов). Пиши только текст письма — без пояснений "
    "и комментариев. Не добавляй ничего после письма. Не используй "
    "Markdown-разметку: никаких **, ##, списков и другого форматирования."
)

CONTEXT_PROMPT = (
    "Вот примеры реальных писем одного класса:\n\n{examples}\n\n"
    "Опиши одним абзацем, о чём обычно эти письма: тематика, тип обращения, "
    "характерные особенности. Только описание, без вступлений."
)

GEN_PROMPT = (
    "Класс писем: «{class_name}»\n\n"
    "Характеристика класса:\n{context}\n\n"
    "Примеры реальных писем этого класса:\n\n{examples}\n\n"
    "Напиши одно новое письмо для класса «{class_name}». "
    "Сохрани плейсхолдеры вида [PERSON], [ORGANIZATION], [DATE_TIME] "
    "в неизменном виде. Пиши только текст письма."
)


def llm_chat_batch(user_prompts, sampling_params, system_prompt=SYSTEM_PROMPT):
    """Батчевая генерация: все промпты летят на GPU параллельно."""
    conversations = [
        [{"role": "system", "content": system_prompt},
         {"role": "user",   "content": up}]
        for up in user_prompts
    ]
    outputs = llm.chat(conversations, sampling_params)
    return [o.outputs[0].text.strip() if o.outputs else "" for o in outputs]

## 5. Фильтры валидации

In [6]:
from langdetect import detect, LangDetectException
import os
os.environ["TRANSFORMERS_NO_TORCHCODEC"] = "1"
import sys
sys.modules["torchcodec"] = None
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("Загружаю SBERT (CPU)...")
sbert = SentenceTransformer("ai-forever/sbert_large_nlu_ru", device="cpu")

CJK_RE = re.compile(r"[\u4e00-\u9fff\u3040-\u309f\u30a0-\u30ff\uac00-\ud7af]")
LEAK_MARKERS = [
    "конечно,", "конечно!", "вот письмо", "вот пример", "вот несколько",
    "пример письма", "напиши одно", "переформулирован", "как языковая модель",
    "я создам", "для класса «", "предоставь пример", "вот ещё одно",
]

def is_truncated(text):
    t = text.rstrip()
    if not t:
        return True
    return t[-1] not in ".!?)\"»]"

def is_degenerate(text):
    words = text.lower().split()
    if len(words) < 3:
        return True
    if len(set(words)) / len(words) < 0.2:
        return True
    if re.search(r"(\b\w+(?:\s+\w+){2,})\s+\1\s+\1", text.lower()):
        return True
    return False

def is_prompt_leak(text):
    low = text.strip().lower()
    if any(m in low[:150] for m in LEAK_MARKERS):
        return True
    if re.search(r"\*\*[^*\n]{3,}?\*\*|(?:^|\n)###?\s", text, re.MULTILINE):
        return True
    return False

def is_russian(text):
    try:
        return detect(text) == "ru"
    except LangDetectException:
        return False


def validate_batch(candidates, existing_texts):
    seen = {t.strip().lower() for t in existing_texts}
    passed = []
    for t in candidates:
        if not t or len(t.strip()) < MIN_LENGTH:
            continue
        norm = t.strip().lower()
        if norm in seen:
            continue
        if is_truncated(t) or is_degenerate(t) or is_prompt_leak(t):
            continue
        if CJK_RE.search(t) or not is_russian(t):
            continue
        seen.add(norm)
        passed.append(t)
    if not passed or not existing_texts:
        return passed
    emb_new = sbert.encode(passed, show_progress_bar=False)
    emb_old = sbert.encode(existing_texts, show_progress_bar=False)
    sims = cosine_similarity(emb_new, emb_old).max(axis=1)
    return [t for t, s in zip(passed, sims) if SIM_LOW <= s < SIM_HIGH]

Загружаю SBERT (CPU)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

## 6. Аугментация класса (батчами)

In [7]:
def make_examples(texts, k=MAX_EXAMPLES_IN_PROMPT, max_chars=1500):
    sample = random.sample(texts, min(k, len(texts)))
    trimmed = [t[:max_chars] for t in sample]
    return "\n---\n".join(trimmed)


def augment_class(class_name, existing, n_needed):
    # описание класса — один батч из 1 промпта
    ctx_raw = llm_chat_batch(
        [CONTEXT_PROMPT.format(examples=make_examples(existing))],
        ctx_params, system_prompt="Ты аналитик деловой переписки.",
    )[0]
    ctx = ctx_raw.split("\n\n")[0].strip()

    collected, pool = [], list(existing)
    for attempt in range(1, MAX_RETRIES + 1):
        need = n_needed - len(collected)
        if need <= 0:
            break
        batch_n = need * OVERSAMPLE + 1
        # формируем батч промптов — у каждого свой набор примеров
        prompts = [
            GEN_PROMPT.format(class_name=class_name, context=ctx, examples=make_examples(pool))
            for _ in range(batch_n)
        ]
        # ВЕСЬ батч параллельно через vLLM
        cands = llm_chat_batch(prompts, gen_params)
        valid = validate_batch(cands, pool)
        take = valid[:need]
        collected.extend(take); pool.extend(take)
        print(f"  попытка {attempt}: сгенерировано {len(cands)}, прошло {len(valid)}, "
              f"взято {len(take)}, всего {len(collected)}/{n_needed}")
    return collected

## 7. Запуск этапа 1

In [8]:
counts = df["label"].value_counts()
classes_to_aug = counts[counts < TARGET_COUNT].sort_values()
print(f"Классов для аугментации: {len(classes_to_aug)}\n")

new_rows = []
for i, (cls, cur) in enumerate(classes_to_aug.items(), 1):
    need = TARGET_COUNT - cur
    existing = df[df["label"] == cls]["text"].tolist()
    print(f"[{i}/{len(classes_to_aug)}] «{cls[:40]}»: есть {cur}, нужно ещё {need}")
    gen = augment_class(cls, existing, need)
    for t in gen:
        new_rows.append({"label": cls, "text": t, "source": "stage1_llm_gen"})
    tmp = pd.concat([df.assign(source="original"), pd.DataFrame(new_rows)], ignore_index=True)
    tmp.to_csv(OUT_PATH, index=False)
    print(f"  добавлено {len(gen)}, промежуточно сохранено {len(tmp)}\n")

print(f"Этап 1 завершён. Добавлено {len(new_rows)} писем.")

Классов для аугментации: 18

[1/18] «Имущественные вопросы»: есть 1, нужно ещё 19


Rendering conversations:   0%|          | 0/1 [00:00<?, ?it/s]

INFO 05-21 14:12:52 [hf.py:483] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering conversations:   0%|          | 0/39 [00:00<?, ?it/s]

...

Rendering conversations:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering conversations:   0%|          | 0/3 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/3 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  попытка 1: сгенерировано 3, прошло 3, взято 1, всего 1/1
  добавлено 1, промежуточно сохранено 1620

Этап 1 завершён. Добавлено 216 писем.


## 8. Итог

In [9]:
final = pd.read_csv(OUT_PATH)
print(f"Было: {len(df)} | Стало: {len(final)} (+{len(final)-len(df)})")
print(f"\nИсточники:")
print(final["source"].value_counts())
print(f"\nМинимум на класс: {final['label'].value_counts().min()}")

from google.colab import files
files.download(str(OUT_PATH))

Было: 1404 | Стало: 1620 (+216)

Источники:
source
original          1404
stage1_llm_gen     216
Name: count, dtype: int64

Минимум на класс: 20


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>